# Data Preparation

In [0]:
import pandas as pd

# Import our synthetic dataset from GitHub repo
raw_url = "https://raw.githubusercontent.com/jazz1416/VetTrack-Customer-Service-Agent/refs/heads/main/synthetic_customer_support_tickets.csv"

df = pd.read_csv(raw_url)

df.head()

### Dataset Overview

In [0]:
# Print number of rows and columns in dataset
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [0]:
# Display column types
df.info()

In [0]:
# Summary stats
df.describe()

###  Preprocessing

In [0]:
# Get columns that contain n/a values
df.columns[df.isna().any()].tolist()

In [0]:
# Fill na values that can easily be filled
df['Resolution'] = df['Resolution'].fillna("No resolution provided")
# Fill empty customer satisfaction ratings with the median value 
df['Customer Satisfaction Rating'] = df['Customer Satisfaction Rating'].fillna(3)
df

In [0]:
import pandas as pd

# Cast columns to proper data types
df['Ticket ID'] = df['Ticket ID'].astype('int64')
df['Customer Name'] = df['Customer Name'].astype('string')
df['Customer Email'] = df['Customer Email'].astype('string')
df['Customer Age'] = df['Customer Age'].astype('int64')
df['Customer Gender'] = df['Customer Gender'].astype('string')
df['Product Purchased'] = df['Product Purchased'].astype('string')
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'])
df['Ticket Type'] = df['Ticket Type'].astype('string')
df['Ticket Subject'] = df['Ticket Subject'].astype('string')
df['Ticket Description'] = df['Ticket Description'].astype('string')
df['Ticket Status'] = df['Ticket Status'].astype('string')
df['Resolution'] = df['Resolution'].astype('string')
df['Ticket Priority'] = df['Ticket Priority'].astype('string')
df['Ticket Channel'] = df['Ticket Channel'].astype('string')
df['First Response Time'] = pd.to_datetime(df['First Response Time'])
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'])
df['Customer Satisfaction Rating'] = df['Customer Satisfaction Rating'].astype('int64')
df['Minutes to resolve'] = df['Minutes to resolve'].astype('Int64')

In [0]:
# Ensure that columns are the right data types
df.info()

In [0]:
import re

# Normalize text
def normalize_text(text):
    """
    Standardizes text for the agent to read better
    """
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

# Normalize text for necessary columns
df['Ticket Description'] = df['Ticket Description'].apply(normalize_text)
df['Resolution'] = df['Resolution'].apply(normalize_text)

In [0]:
cat_cols = ['Customer Gender', 'Product Purchased', 'Ticket Type', 'Ticket Subject', 'Ticket Status', 'Ticket Priority', 'Ticket Channel']

# Create a new column with int values for each category 
for col in cat_cols:
    numerical_values, _ = pd.factorize(df[col])
    current_index = df.columns.get_loc(col)
    new_col_name = f'{col} ID'
    df.insert(current_index + 1, column = new_col_name, value = numerical_values)

In [0]:
import numpy as np

# Create new column that indicates if a ticket has yet to be picked up 
# due to a lack of information
df['Awaiting Additional Information'] = np.where(df['First Response Time'].isna(), 'yes', 'no')

In [0]:
pip install sentence-transformers

In [0]:
# Create embeddings with batch processing
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Store ticket descriptions in list for embeddings
texts = df['Ticket Description'].tolist()

# Process embeddings in batches
batch_size = 1000
embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True)

# Create new column in df to store embeddings
df['embeddings'] = list(embeddings)
print(f"Created embeddings with shape: {embeddings.shape}")

In [0]:
# Create the default catalog and schema if they don't already exist
spark.sql("CREATE CATALOG IF NOT EXISTS main")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.default")

In [0]:
# Unity Catalog location: catalog.schema.table
catalog = "main"
schema = "default"
table_name = "final_project"

# Rename columns to replace spaces with underscores 
df.columns = df.columns.str.replace(' ', '_')

# Convert Pandas DataFrame to Spark and write as Delta
# mode("overwrite") replaces the table if it exists (good for re-running the lab)
final_project = spark.createDataFrame(df)
final_project.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table_name}")

# Verify: query the table
display(spark.table(f"{catalog}.{schema}.{table_name}").limit(5))

# Exploratory Data Analysis

### Feature Analysis

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

categorical_cols = ['Customer_Gender', 'Ticket_Type', 'Ticket_Subject', 'Ticket_Status', 'Ticket_Priority', 'Ticket_Channel', 
                    'Awaiting_Additional_Information']

# Plot categorical columns as bar graphs
for col in categorical_cols:
    plt.figure(figsize=(24,4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()


In [0]:
# Plot age histogram
plt.figure(figsize=(10,5))
plt.hist(df['Customer_Age'], bins=20, edgecolor='black')
plt.title("Customer Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

### Text analysis

In [0]:
%pip install nltk

In [0]:
from collections import Counter
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# Combine all issue descriptions
all_text = " ".join(df['Ticket_Description'])

# Tokenize
words = [w for w in all_text.split() if w not in stop_words]

# Most common words
word_freq = Counter(words).most_common(20)

# Plot most common words
word_df = pd.DataFrame(word_freq, columns=['word', 'count'])
plt.figure(figsize=(15,5))
plt.bar(word_df['word'], word_df['count'])
plt.xticks(rotation=45)
plt.title("Top 20 Most Common Words in Ticket Descriptions")
plt.show()

### Embedding

In [0]:
from sklearn.decomposition import PCA

# Convert embeddings column to array
emb_matrix = np.vstack(df['embeddings'].values)

# Reduce to 2D
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(emb_matrix)

# Plot
plt.figure(figsize=(10,7))
plt.scatter(emb_2d[:,0], emb_2d[:,1], s=5, alpha=0.5)
plt.title("PCA Visualization of Ticket Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()


In [0]:
from sklearn.cluster import KMeans

pca = PCA(n_components=10).fit_transform(emb_matrix)

# Cluster embeddings
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(pca)

# Visualize clusters
plt.figure(figsize=(10,7))
sns.scatterplot(x=pca[:,0], y=pca[:,1], hue=df['cluster'], palette='tab10', s=10)
plt.title("Ticket Embedding Clusters")
plt.show()

# Gather most common words for each cluster
def top_words_for_cluster(cluster_id, n=20):
    texts = df[df['cluster'] == cluster_id]['Ticket_Description']
    all_words = " ".join(texts).split()
    common = Counter(all_words).most_common(n)
    return common

# Print top words for each cluster
for c in sorted(df['cluster'].unique()):
    print(f"\nCluster {c} top words:")
    print(top_words_for_cluster(c))

# Gather sample tickets for each cluster
def sample_tickets(cluster_id, n=5):
    return df[df['cluster'] == cluster_id]['Ticket_Description'].head(n)

# Print sample tickets for each cluster
for c in sorted(df['cluster'].unique()):
    print(f"\nCluster {c} sample tickets:")
    print(sample_tickets(c))

In [0]:
# Group clusters by priority
cluster_high_priority = (
    df.groupby('cluster')
      .apply(lambda x: (x['Ticket_Priority'].isin(['High', 'Critical'])).mean(), include_groups=False)
      .sort_values()
)
print("High Priority Rate by Cluster:")
print(cluster_high_priority)

if 'Customer Satisfaction Rating' in df.columns:
    cluster_satisfaction = (
        df.groupby('cluster')['Customer_Satisfaction_Rating']
          .mean()
          .sort_values()
    )
    print("\nMean Customer Satisfaction Rating by Cluster (1-5):")
    print(cluster_satisfaction)